In [ ]:
!pip install datasets

In [ ]:
# === ✅ Install All Required Packages ===
!pip install -U transformers accelerate peft mlflow datasets

In [ ]:
!pip uninstall -y bitsandbytes
!pip install --upgrade bitsandbytes

Found existing installation: bitsandbytes 0.46.0
Uninstalling bitsandbytes-0.46.0:
  Successfully uninstalled bitsandbytes-0.46.0
  Using cached bitsandbytes-0.46.0-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.0-py3-none-manylinux_2_24_x86_64.whl (67.0 MB)


In [ ]:
import bitsandbytes as bnb

# Test GPU module availability
import bitsandbytes.nn as bnb_nn
_ = bnb_nn.Linear8bitLt(128, 128)
print("✅ bitsandbytes 8-bit GPU module is working")

✅ bitsandbytes 8-bit GPU module is working


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# === ✅ Import Modules ===
import os
import pandas as pd
import torch
import mlflow
from torch.utils.data import Dataset
from transformers import (
    LlamaForCausalLM, LlamaTokenizerFast,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import Dataset

In [ ]:
# === ✅ Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# === ✅ Paths ===
MERGED_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2_MERGED"
BASE_MODEL = "meta-llama/Llama-2-7b-hf"
BASE_MODEL_SAVE_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit"
LORA_ADAPTER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter"
OUTPUT_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_training_output"

In [ ]:
# === ✅ Load Tokenizer ===
tokenizer = LlamaTokenizerFast.from_pretrained(MERGED_TOKENIZER_PATH)
tokenizer.add_special_tokens({'pad_token': '<pad>'})
tokenizer.pad_token = '<pad>'

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizer'. 
The class this function is called from is 'LlamaTokenizerFast'.


In [ ]:
habibi = load_dataset("arbml/Habibi", split="train")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
tashkeela = load_dataset("EmanKhater/Tashkeela", split="train")

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
arabicpile_social = load_dataset("premio-ai/TheArabicPile_SocialMedia", split="original", streaming=True)

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

In [ ]:
brad = load_dataset("arbml/BRAD", split="train")

In [ ]:
from datasets import Dataset

# === ✅ 3. Load Dataset: TheArabicPile_SocialMedia (streamed, convert to memory)
arabicpile_social = load_dataset("premio-ai/TheArabicPile_SocialMedia", split="original", streaming=True)

# Convert to in-memory (limit samples to avoid Colab crash, increase later)
social_texts = []
for i, row in enumerate(arabicpile_social):
    if "text" in row and row["text"].strip():
        social_texts.append({"cleaned_text": row["text"]})
    if i >= 500_000:  # Adjust or remove this limit based on your RAM
        break

social_ds = Dataset.from_list(social_texts)

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

In [ ]:
# ✅ Rename all columns to unified key
habibi = habibi.rename_column("Lyrics", "cleaned_text")
tashkeela = tashkeela.rename_column("text", "cleaned_text")
brad = brad.rename_column("review", "cleaned_text")

In [ ]:
def verify_column(dataset, name):
    if "cleaned_text" not in dataset.column_names:
        print(f"❌ '{name}' is missing 'cleaned_text' column!")
    else:
        print(f"✅ '{name}' includes 'cleaned_text' column with {len(dataset)} samples")

verify_column(habibi, "Habibi")
verify_column(tashkeela, "Tashkeela")
verify_column(social_ds, "SocialMedia")
verify_column(brad, "BRAD")

✅ 'Habibi' includes 'cleaned_text' column with 527896 samples
✅ 'Tashkeela' includes 'cleaned_text' column with 3090000 samples
✅ 'SocialMedia' includes 'cleaned_text' column with 500001 samples
✅ 'BRAD' includes 'cleaned_text' column with 510598 samples


In [ ]:
# ✅ Combine datasets
combined_dataset = concatenate_datasets([habibi, tashkeela, social_ds, brad])

In [ ]:
dataset = combined_dataset.shuffle(seed=42).select(range(100_000))

In [ ]:
print("✅ Habibi size:", len(habibi))
print("✅ Tashkeela size:", len(tashkeela))
print("✅ Social Media chunk size:", len(social_ds))
print("✅ BRAD size:", len(brad))

✅ Habibi size: 527896
✅ Tashkeela size: 3090000
✅ Social Media chunk size: 500001
✅ BRAD size: 510598


In [ ]:
print("✅ Combined size:", len(combined_dataset))

✅ Combined size: 4628495


In [ ]:
print("✅ Combined size:", len(dataset))

✅ Combined size: 100000


In [ ]:
texts = combined_dataset["cleaned_text"]
unique_texts = set(texts)
print(f"❗ Duplicate entries: {len(texts) - len(unique_texts)}")

❗ Duplicate entries: 122781


In [ ]:
texts = dataset["cleaned_text"]
unique_texts = set(texts)
print(f"❗ Duplicate entries: {len(texts) - len(unique_texts)}")

❗ Duplicate entries: 142


In [ ]:
for i in range(10):
    print(f"{i+1}.", combined_dataset[i]["cleaned_text"])

1. اروح لاحبابي والاقي الفرح ساكن عينهم
2. ابتسم لافراحهم وانا من الهم احترق
3. واسأل جروحي من ترى حس بعذابي منهم
4. وبالحقيقه انصدم محدن معه همي فرق
5. دورت في كل الوجيه حسيت غربه بينهم
6. مع الأسف محدن ابد حس بعذاباتي ورق
7. جيت اتعثر بالتعب ابي اشوف يدينهم
8. ماكنت ابي الا احد يحس بي لو مانطق
9. وحز فيني اني رجعت لكن رجعت بدونهم
10. يحز في نفسي بأنه ماسوى جرحي صدق


In [ ]:
# === ✅ Tokenization Dataset
class ArabicDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=384):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["cleaned_text"]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": labels.squeeze(0)
        }

In [ ]:
train_dataset = ArabicDataset(dataset, tokenizer)

In [ ]:
# === ✅ Sanity check ===
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

input_ids: shape=torch.Size([384]), dtype=torch.int64
attention_mask: shape=torch.Size([384]), dtype=torch.int64
labels: shape=torch.Size([384]), dtype=torch.int64


In [ ]:
# === ✅ Quant Config ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# === ✅ Save clean quantized base model FIRST
base_model = LlamaForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.save_pretrained(BASE_MODEL_SAVE_PATH, safe_serialization=True)
tokenizer.save_pretrained(BASE_MODEL_SAVE_PATH)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

('/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit/tokenizer_config.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit/special_tokens_map.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit/tokenizer.json')

In [ ]:
# === ✅ Reload and Prepare Model
model = LlamaForCausalLM.from_pretrained(
    BASE_MODEL_SAVE_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.11/dist-packages/transformers/quantizers/auto.py:222: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


In [ ]:
# === ✅ Fresh LoRA Adapter Config
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(94948, 4096)

In [ ]:
layer = model.model.model.layers[0].self_attn.q_proj
print(type(layer))                               # ✅ bitsandbytes.nn.Linear4bit
print(type(layer.weight))                        # ✅ bitsandbytes.nn.Params4bit
print(hasattr(layer.weight, "quant_type"))       # ✅ True
print(hasattr(layer.weight, "compress_statistics"))  # ✅ True

<class 'peft.tuners.lora.bnb.Linear4bit'>
<class 'bitsandbytes.nn.modules.Params4bit'>
True
True


In [ ]:
# === ✅ MLflow Tracking
mlflow.set_experiment("LLaMA7B_Arabic_Adaptive_Training")

<Experiment: artifact_location='file:///content/mlruns/170947057427111297', creation_time=1751033104514, experiment_id='170947057427111297', last_update_time=1751033104514, lifecycle_stage='active', name='LLaMA7B_Arabic_Adaptive_Training', tags={}>

In [ ]:
# === ✅ TrainingArguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=2,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"]
)

In [ ]:
# === ✅ MLflow Tracking
mlflow.set_experiment("Adaptive_LLaMA7B_Arabic_100k")
mlflow.log_params({
    "model": BASE_MODEL,
    "tokenizer": MERGED_TOKENIZER_PATH,
    "num_samples": 100_000,
    "epochs": 2,
    "batch_size": 4 * 4,
    "learning_rate": 5e-5
})

In [ ]:
# === ✅ Split train/validation
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = ArabicDataset(split["train"], tokenizer)
eval_dataset = ArabicDataset(split["test"], tokenizer)

In [ ]:
# === ✅ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset  # ✅ Required for evaluation
)

In [ ]:
from datasets import DatasetDict

# Split dataset 95% train, 5% eval
split_ds = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = ArabicDataset(split_ds["train"], tokenizer)
eval_dataset = ArabicDataset(split_ds["test"], tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset  # ✅ This enables eval loss tracking
)

In [ ]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k}: shape={v.shape} | type={v.dtype}")

input_ids: shape=torch.Size([384]) | type=torch.int64
attention_mask: shape=torch.Size([384]) | type=torch.int64
labels: shape=torch.Size([384]) | type=torch.int64


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,7.133500,7.501228
2,6.897200,7.478939


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=11876, training_loss=7.262357450647762, metrics={'train_runtime': 38635.9085, 'train_samples_per_second': 4.918, 'train_steps_per_second': 0.307, 'total_flos': 3.01998938259456e+18, 'train_loss': 7.262357450647762, 'epoch': 2.0})

In [ ]:
# === ✅ Save LoRA Adapter
model.save_pretrained(LORA_ADAPTER_PATH)
tokenizer.save_pretrained(LORA_ADAPTER_PATH)

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter/special_tokens_map.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter/tokenizer.json')

In [ ]:
# === ✅ Finish
mlflow.end_run()